## <font color=DarkTurquoise>Import packages</font>

In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import nlp
import tensorflow as tf
import random
import os
%matplotlib inline
import re
from transformers import BertTokenizer
import tensorflow as tf
from transformers import TFBertForSequenceClassification
from sklearn.model_selection import train_test_split

## <font color=DarkTurquoise>Dataset</font>

We obtain the dataset from [this Kaggle link](https://www.kaggle.com/datasets/parulpandey/emotion-dataset/data), which is a dataset of English Twitter messages categorized into five basic emotions: sadness (0), joy (1), love (2), anger (3), and fear (4). We will use this dataset to train a BERT model and fine-tune it to perform emotion recognition on our movie plot summaries.


In [ ]:
'''# Load the training/validation/testing dataset from the specified file path
train = pd.read_csv('data/data_for_training/training.csv')
val = pd.read_csv('data/data_for_training/validation.csv')
test = pd.read_csv('data/data_for_training/test.csv')
train'''

In [ ]:
data = pd.read_csv('../../data/tweet_emotions/tweet_emotions.csv')
data['sentiment'] = data['sentiment'].replace('empty', 'neutral')
label_mapping = {label: idx for idx, label in enumerate(data['sentiment'].unique())}
data['sentiment_label'] = data['sentiment'].map(label_mapping)
print("Label Mapping:", label_mapping)
data

In [ ]:
df = data[['content', 'sentiment_label']].rename(columns={'content': 'text', 'sentiment_label': 'label'})
train, temp = train_test_split(df, test_size=0.2, random_state=42)
val, test = train_test_split(temp, test_size=0.5, random_state=42)
test

In [ ]:
# Load a pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize the 'text' column from the training/validation dataset
'''train_encodings = tokenizer(list(train['text']), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val['text']), truncation=True, padding=True, max_length=128)'''

train_encodings = tokenizer(list(train['text']), truncation=False, padding=True)
val_encodings = tokenizer(list(val['text']), truncation=False, padding=True)


In [ ]:
# Create a TensorFlow Dataset for training/validation data
train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings), 
    train['label']         
))

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings), 
    val['label']          
))



In [ ]:
# Load a pre-trained BERT model for sequence classification
model = TFBertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=12)  

After three epochs, the model achieved a loss of 0.1232, accuracy of 94.27%, validation loss of 0.1704, and validation accuracy of 93.35%. We saved the model parameters and tested its performance.

In [ ]:
# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# Train the model
model.fit(
    train_dataset.shuffle(1000).batch(16),
    validation_data=val_dataset.batch(16),
    epochs=3
)


We randomly input several sentences to test the performance of the current model.

In [ ]:
# Example of unlabeled data
unlabeled_texts = ["I feel like everything is falling apart.", "I can't stop smiling; everything feels perfect.",
                "You mean the world to me, and I cherish every moment with you.","This is completely unacceptable; I am furious.",
                "The thought of losing everything keeps me awake at night."]

# Encode the unlabeled data
unlabeled_encodings = tokenizer(unlabeled_texts, truncation=True, padding=True, max_length=128, return_tensors="tf")

# Predict the labels for the unlabeled data
predictions = model.predict(unlabeled_encodings.data)
predicted_labels = tf.argmax(predictions.logits, axis=1).numpy()

# Print the predicted labels alongside their corresponding texts
for text, label in zip(unlabeled_texts, predicted_labels):
    print(f"Text: {text}, Predicted Label: {label}")



In [ ]:
# Save the model
model.save_pretrained("../model/bert_finetuned_model_1")
tokenizer.save_pretrained("../model/bert_finetuned_model_1")
